# Φ_l factory — batch modular polynomial production

**Purpose**: compute the classical modular polynomials Φ_l for every prime l ≤ 71 missing from
`pycode/data/classical_modpolys.json` — in particular the non-Atkin primes **37, 43, 53, 61, 67** —
on a machine that can run unattended for a long time.

**Naming**: l is the isogeny degree (we compute Φ_l); p is the characteristic being interpolated at.

**Just hit Run All.** Everything checkpoints (the rigid-l-set cache per disc, the CRT state in
`data/phi_factory_state.json` after every characteristic, each finished Φ_l immediately into
`classical_modpolys.json`), so you can interrupt and re-run at any point; finished targets are skipped.

**What it does per characteristic p** (ascending, starting from the shipped p < 1024 data — and
reusing the bij_factory store wherever it covers p): extend the rigid-l-set cache; compute the
j ↔ form bijections of every class at p; extract l-isogenous pairs for each unfinished target;
solve the linear system for Φ_l mod p (diagonal = −∏H_d from the Hilbert library + pair
evaluations + any special-value rows the library supports); CRT-accumulate. A target finishes when
its modulus exceeds the provable Bröker–Sutherland height bound, passes the Kronecker congruence
check, and is saved.

**Expected behaviour**: each target's banner prints its predicted characteristic range from
`phi_prime_range_estimate` (full-rank threshold + prime number theorem vs the B-S height;
calibrated within ~6% on l = 29, 31, 37). Small targets finish first; Φ_71 should finish around
p ≈ 9000. Each FINISHED line reports the actual range used.

**Getting results back**: commit `data/classical_modpolys.json`, `data/rigid_lset_cache.json`,
and `data/phi_factory_state.json`.

In [1]:
import os
os.chdir('../pycode')
from phi_factory import *

# optional GPU trace tables (needs pytorch; falls back to numpy silently)
try:
    import trace_gpu
    print('trace backend:', trace_gpu.enable())
except ImportError:
    print('trace backend: numpy (pip install torch to enable cuda/mps)')

targets = factory_targets()
print('missing modular polynomials for p =', targets)
print('priority (non-Atkin):', [p for p in targets if p in WANTED])

trace backend: mps
missing modular polynomials for p = [29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71]
priority (non-Atkin): [37, 43, 53, 61, 67]


In [2]:
# preflight: the diagonal support Hilbert polynomials must all be known
from modpoly_crt import phi_diagonal
for p in targets:
    phi_diagonal(p)          # raises with the missing discs if the library falls short
print('diagonal support: complete for all targets')

diagonal support: complete for all targets


In [ ]:
# the long run -- safe to interrupt and re-run at any time
run_factory(targets)

target p=29: 465 unknowns, bound 1881 bits, 0 bits done, 1 special-value families
target p=31: 528 unknowns, bound 2023 bits, 0 bits done, 1 special-value families
target p=37: 741 unknowns, bound 2454 bits, 0 bits done, 0 special-value families
target p=41: 903 unknowns, bound 2745 bits, 0 bits done, 0 special-value families
target p=43: 990 unknowns, bound 2891 bits, 0 bits done, 0 special-value families
target p=47: 1176 unknowns, bound 3184 bits, 0 bits done, 0 special-value families
target p=53: 1485 unknowns, bound 3629 bits, 0 bits done, 0 special-value families
target p=59: 1830 unknowns, bound 4077 bits, 0 bits done, 0 special-value families
target p=61: 1953 unknowns, bound 4227 bits, 0 bits done, 0 special-value families
target p=67: 2346 unknowns, bound 4680 bits, 0 bits done, 0 special-value families
target p=71: 2628 unknowns, bound 4984 bits, 0 bits done, 0 special-value families
ell=773 (1s): p=29 10/1881
ell=797 (1s): p=29 19/1881
ell=823 (1s): p=29 29/1881
ell=839 (1s

In [ ]:
# quick sanity view of whatever has finished so far
from modularpolynomials import _modpoly_cache
from modpoly_crt import kronecker_check
done = sorted(p for p in targets if p in _modpoly_cache)
for p in done:
    print(f'Phi_{p}: kronecker mod p:', kronecker_check(p, _modpoly_cache[p]))
print('finished:', done)

In [ ]:
# actual characteristic ranges vs predictions
factory_report()